# CertGen CVPR Checkpoint Preflight — Kaggle T4x2

`non_evidence_preflight` · `not_empirical_evidence` · `not paper evidence` · `claim_allowed=false`

Production-hardened, static-validation passed, fixture-runtime passed; real Kaggle preflight is still required. The run contract is hash-bound and supports `resume`, `restart`, and `force_new_run`. GPU work occurs only in isolated subprocess workers.

Select **GPU T4 ×2** in Kaggle before running any code. Kaggle Internet is used only according to the immutable dependency mode; model and extractor loading remains offline from a validated private mount.

This notebook uses multiprocessing `spawn`, keeps CUDA out of the parent, and schedules one subprocess worker per physical GPU.


## 0 Human instructions

Run top-to-bottom only after selecting `GPU T4 ×2`. Never edit a frozen hash in place.


## 1 Immutable user configuration


## 2 Input discovery


In [ ]:
from __future__ import annotations
import hashlib, json, os, subprocess, sys
from pathlib import Path

SEARCH_ROOTS = [Path(value) for value in os.environ.get("CERTGEN_SEARCH_ROOTS", "/kaggle/input:/kaggle/working").split(os.pathsep) if value]
_TRUSTED_BOOTSTRAP_SOURCE = '"""Stdlib-only authentication gate used before any package import in notebooks.\n\nThis module deliberately imports only the Python standard library.  Notebook\ngeneration embeds this file\'s exact source and its SHA-256; the embedded code\ndiscovers, authenticates, and atomically materializes a package before the\nmaterialized directory is added to ``sys.path``.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\n\nDEFAULT_LIMITS = {\n    "maximum_depth": 12,\n    "maximum_candidates": 10_000,\n    "maximum_package_members": 200_000,\n    "maximum_uncompressed_bytes": 20 * 1024**3,\n    "maximum_metadata_bytes": 4 * 1024**2,\n    "maximum_compression_ratio": 1_000.0,\n}\nRUNTIME_MARKERS = {".source_sha256", ".certgen_runtime_location.json"}\nNESTED_ARCHIVE_SUFFIXES = (\n    ".zip",\n    ".tar",\n    ".tgz",\n    ".tar.gz",\n    ".tbz",\n    ".tbz2",\n    ".7z",\n    ".rar",\n)\nEXPECTED_IDENTITY_FIELDS = (\n    "expected_package_sha256",\n    "expected_scientific_identity_hash",\n    "expected_configuration_hash",\n    "expected_run_id",\n    "expected_study_hash",\n    "expected_profile_id",\n    "expected_scale",\n    "expected_source_code_hash",\n    "expected_integrity_manifest",\n    "expected_output_schema_version",\n)\n\n\nclass AuthenticationError(RuntimeError):\n    """The candidate did not satisfy the pre-import authentication contract."""\n\n\ndef _sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef _sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _content_hash(members: dict[str, bytes]) -> str:\n    digest = hashlib.sha256()\n    for name, data in sorted(members.items()):\n        digest.update(name.encode("utf-8"))\n        digest.update(b"\\0")\n        digest.update(hashlib.sha256(data).digest())\n    return digest.hexdigest()\n\n\ndef _json_object(data: bytes, name: str, maximum_metadata_bytes: int) -> dict:\n    if len(data) > maximum_metadata_bytes:\n        raise AuthenticationError(f"oversized metadata rejected: {name}")\n    try:\n        value = json.loads(data.decode("utf-8"))\n    except (UnicodeDecodeError, json.JSONDecodeError) as exc:\n        raise AuthenticationError(f"invalid JSON metadata: {name}") from exc\n    if not isinstance(value, dict):\n        raise AuthenticationError(f"metadata is not an object: {name}")\n    return value\n\n\ndef _safe_member_name(raw: str) -> str:\n    member = PurePosixPath(raw)\n    if not raw or member.is_absolute() or ".." in member.parts or "\\\\" in raw:\n        raise AuthenticationError(f"unsafe package member: {raw}")\n    return member.as_posix()\n\n\ndef _inspect_zip(path: Path, limits: dict) -> tuple[dict[str, bytes], str]:\n    package_sha256 = _sha256_file(path)\n    try:\n        archive = zipfile.ZipFile(path)\n    except (OSError, zipfile.BadZipFile) as exc:\n        raise AuthenticationError(f"unreadable package ZIP: {exc}") from exc\n    with archive:\n        infos = archive.infolist()\n        if len(infos) > limits["maximum_package_members"]:\n            raise AuthenticationError("package member-count limit exceeded")\n        seen: set[str] = set()\n        total = 0\n        for info in infos:\n            name = _safe_member_name(info.filename)\n            key = name.casefold()\n            if key in seen:\n                raise AuthenticationError(f"duplicate or case-colliding package member: {name}")\n            seen.add(key)\n            mode = (info.external_attr >> 16) & 0o177777\n            file_type = stat.S_IFMT(mode)\n            if file_type == stat.S_IFLNK:\n                raise AuthenticationError(f"symlink archive entry rejected: {name}")\n            if file_type not in {0, stat.S_IFREG, stat.S_IFDIR}:\n                raise AuthenticationError(f"hard-link or special archive entry rejected: {name}")\n            lowered = name.casefold()\n            if not info.is_dir() and lowered.endswith(NESTED_ARCHIVE_SUFFIXES):\n                raise AuthenticationError(f"nested archive rejected: {name}")\n            total += info.file_size\n            if total > limits["maximum_uncompressed_bytes"]:\n                raise AuthenticationError("package uncompressed-byte limit exceeded")\n            if info.file_size and info.compress_size == 0:\n                raise AuthenticationError(f"invalid compressed size: {name}")\n            ratio = info.file_size / max(info.compress_size, 1)\n            if ratio > limits["maximum_compression_ratio"]:\n                raise AuthenticationError(f"compression-ratio limit exceeded: {name}")\n        if archive.testzip() is not None:\n            raise AuthenticationError("package CRC validation failed")\n        members = {\n            info.filename: archive.read(info.filename)\n            for info in infos\n            if not info.is_dir()\n        }\n    return members, package_sha256\n\n\ndef _inspect_directory(path: Path, limits: dict) -> tuple[dict[str, bytes], str]:\n    members: dict[str, bytes] = {}\n    total = 0\n    for item in sorted(path.rglob("*")):\n        relative = item.relative_to(path).as_posix()\n        depth = len(PurePosixPath(relative).parts)\n        if depth > limits["maximum_depth"]:\n            raise AuthenticationError(f"extracted package depth limit exceeded: {relative}")\n        if item.is_symlink():\n            raise AuthenticationError(f"symlink in extracted package rejected: {relative}")\n        try:\n            item_stat = item.stat()\n        except OSError as exc:\n            raise AuthenticationError(f"extracted package entry is unreadable: {relative}") from exc\n        if item.is_dir():\n            continue\n        if not item.is_file() or item_stat.st_nlink != 1:\n            raise AuthenticationError(f"hard link or special extracted entry rejected: {relative}")\n        if relative in RUNTIME_MARKERS:\n            continue\n        _safe_member_name(relative)\n        if relative.casefold().endswith(NESTED_ARCHIVE_SUFFIXES):\n            raise AuthenticationError(f"nested archive rejected: {relative}")\n        if len(members) >= limits["maximum_package_members"]:\n            raise AuthenticationError("package member-count limit exceeded")\n        total += item_stat.st_size\n        if total > limits["maximum_uncompressed_bytes"]:\n            raise AuthenticationError("package uncompressed-byte limit exceeded")\n        members[relative] = item.read_bytes()\n    marker = path / ".source_sha256"\n    source_sha256 = marker.read_text(encoding="utf-8").strip() if marker.is_file() else ""\n    if source_sha256 and (len(source_sha256) != 64 or any(character not in "0123456789abcdef" for character in source_sha256)):\n        raise AuthenticationError("invalid extracted-package source SHA-256 marker")\n    return members, source_sha256 or _content_hash(members)\n\n\ndef _verify_integrity(members: dict[str, bytes], identity: dict, limits: dict) -> tuple[dict, str]:\n    integrity_name = str(identity.get("integrity_manifest") or "")\n    if not integrity_name or integrity_name not in members:\n        raise AuthenticationError("declared integrity manifest is missing")\n    integrity = _json_object(members[integrity_name], integrity_name, limits["maximum_metadata_bytes"])\n    if integrity.get("claim_allowed") is not False:\n        raise AuthenticationError("integrity manifest must set claim_allowed=false")\n    rows = integrity.get("files")\n    if not isinstance(rows, list) or not rows:\n        raise AuthenticationError("integrity manifest requires non-empty files")\n    declared: dict[str, dict] = {}\n    for index, row in enumerate(rows, start=1):\n        if not isinstance(row, dict):\n            raise AuthenticationError(f"integrity row {index} is not an object")\n        name = _safe_member_name(str(row.get("path") or ""))\n        if name in declared:\n            raise AuthenticationError(f"duplicate integrity path: {name}")\n        declared[name] = row\n        data = members.get(name)\n        if data is None:\n            raise AuthenticationError(f"integrity manifest references absent member: {name}")\n        if row.get("size") != len(data) or row.get("sha256") != _sha256_bytes(data):\n            raise AuthenticationError(f"integrity size/hash mismatch: {name}")\n    actual = set(members) - {integrity_name}\n    if set(declared) != actual:\n        extra = sorted(actual - set(declared))\n        absent = sorted(set(declared) - actual)\n        raise AuthenticationError(\n            "exact package membership mismatch; unexpected="\n            + repr(extra[:20])\n            + "; absent="\n            + repr(absent[:20])\n        )\n    return integrity, integrity_name\n\n\ndef _verify_source_inventory(members: dict[str, bytes], limits: dict) -> tuple[str, list[dict]]:\n    manifest_data = members.get("bundle_manifest.json")\n    if manifest_data is None:\n        raise AuthenticationError("bundle manifest is missing")\n    manifest = _json_object(manifest_data, "bundle_manifest.json", limits["maximum_metadata_bytes"])\n    inventory = manifest.get("source_inventory")\n    if not isinstance(inventory, list) or not inventory:\n        raise AuthenticationError("source-code inventory is missing")\n    digest = hashlib.sha256()\n    seen: set[str] = set()\n    normalized: list[dict] = []\n    for index, row in enumerate(inventory, start=1):\n        if not isinstance(row, dict):\n            raise AuthenticationError(f"source inventory row {index} is not an object")\n        name = _safe_member_name(str(row.get("path") or ""))\n        if not name.startswith("certgen/") or not name.endswith(".py") or name in seen:\n            raise AuthenticationError(f"invalid or duplicate source inventory path: {name}")\n        seen.add(name)\n        data = members.get(name)\n        if data is None:\n            raise AuthenticationError(f"source code absent from package: {name}")\n        observed = _sha256_bytes(data)\n        if row.get("size") != len(data) or row.get("sha256") != observed:\n            raise AuthenticationError(f"source code inventory mismatch: {name}")\n        encoded = name.encode("utf-8")\n        digest.update(len(encoded).to_bytes(8, "big"))\n        digest.update(encoded)\n        digest.update(len(data).to_bytes(8, "big"))\n        digest.update(data)\n        normalized.append({"path": name, "size": len(data), "sha256": observed})\n    actual_sources = {name for name in members if name.startswith("certgen/") and name.endswith(".py")}\n    if actual_sources != seen:\n        raise AuthenticationError(\n            "source-code inventory membership mismatch; unexpected="\n            + repr(sorted(actual_sources - seen)[:20])\n            + "; absent="\n            + repr(sorted(seen - actual_sources)[:20])\n        )\n    observed_hash = digest.hexdigest()\n    if manifest.get("source_code_hash") != observed_hash:\n        raise AuthenticationError("source-code inventory aggregate hash mismatch")\n    return observed_hash, normalized\n\n\ndef _identity_mismatches(identity: dict, expected: dict, package_sha256: str, source_code_hash: str) -> list[str]:\n    mapping = {\n        "expected_package_sha256": package_sha256,\n        "expected_scientific_identity_hash": identity.get("scientific_identity_hash"),\n        "expected_configuration_hash": identity.get("configuration_hash"),\n        "expected_run_id": identity.get("run_id"),\n        "expected_study_hash": identity.get("study_hash"),\n        "expected_profile_id": identity.get("profile_id"),\n        "expected_scale": identity.get("scale"),\n        "expected_source_code_hash": source_code_hash,\n        "expected_integrity_manifest": identity.get("integrity_manifest"),\n        "expected_output_schema_version": identity.get("output_schema_version"),\n        "expected_package_type": identity.get("package_type"),\n        "expected_stage": identity.get("stage"),\n    }\n    mismatches = []\n    for key, observed in mapping.items():\n        required = expected.get(key)\n        if required is not None and required != observed:\n            mismatches.append(f"{key} mismatch: expected={required!r}, observed={observed!r}")\n    return mismatches\n\n\ndef authenticate_candidate(path_value: str | os.PathLike, expected: dict, limits: dict | None = None) -> dict:\n    """Authenticate all bytes and the exact identity of one ZIP/directory."""\n\n    selected_limits = dict(DEFAULT_LIMITS)\n    selected_limits.update(limits or {})\n    path = Path(path_value).resolve(strict=False)\n    if path.is_symlink():\n        raise AuthenticationError("candidate symlink rejected")\n    if path.is_file():\n        members, package_sha256 = _inspect_zip(path, selected_limits)\n        form = "ZIP"\n    elif path.is_dir():\n        members, package_sha256 = _inspect_directory(path, selected_limits)\n        form = "EXTRACTED_DIRECTORY"\n    else:\n        raise AuthenticationError("candidate is not a package ZIP or extracted directory")\n    identity_data = members.get("package_identity.json")\n    if identity_data is None:\n        raise AuthenticationError("package identity is missing")\n    identity = _json_object(\n        identity_data,\n        "package_identity.json",\n        int(selected_limits["maximum_metadata_bytes"]),\n    )\n    if identity.get("claim_allowed") is not False:\n        raise AuthenticationError("package identity must set claim_allowed=false")\n    integrity, integrity_name = _verify_integrity(members, identity, selected_limits)\n    source_code_hash, source_inventory = _verify_source_inventory(members, selected_limits)\n    mismatches = _identity_mismatches(identity, expected, package_sha256, source_code_hash)\n    if mismatches:\n        raise AuthenticationError("exact expected identity rejected: " + "; ".join(mismatches))\n    return {\n        "path": str(path),\n        "form": form,\n        "package_sha256": package_sha256,\n        "identity": identity,\n        "integrity_manifest": integrity_name,\n        "integrity_manifest_sha256": _sha256_bytes(members[integrity_name]),\n        "source_code_hash": source_code_hash,\n        "source_inventory": source_inventory,\n        "member_count": len(members),\n        "uncompressed_bytes": sum(len(data) for data in members.values()),\n        "claim_allowed": False,\n    }\n\n\ndef _candidate_paths(search_roots: list[str | os.PathLike], limits: dict) -> list[Path]:\n    candidates: list[Path] = []\n    skipped = {".git", ".venv", "venv", "node_modules", "__pycache__", ".pytest_cache", ".mypy_cache", ".ruff_cache"}\n    for root_value in search_roots:\n        root = Path(root_value).resolve(strict=False)\n        if not root.exists() or root.is_symlink():\n            continue\n        for current, directories, filenames in os.walk(root, topdown=True, followlinks=False):\n            current_path = Path(current)\n            depth = len(current_path.relative_to(root).parts)\n            directories[:] = sorted(\n                name\n                for name in directories\n                if depth < limits["maximum_depth"]\n                and name not in skipped\n                and not (current_path / name).is_symlink()\n            )\n            if "package_identity.json" in filenames and ".certgen_runtime_location.json" not in filenames:\n                candidates.append(current_path)\n            candidates.extend(\n                current_path / name\n                for name in sorted(filenames)\n                if name.casefold().endswith(".zip") and not (current_path / name).is_symlink()\n            )\n            if len(candidates) > limits["maximum_candidates"]:\n                raise AuthenticationError("bootstrap discovery candidate-count limit exceeded")\n    return sorted(set(candidates), key=lambda value: str(value).casefold())\n\n\ndef discover_authenticated_package(\n    search_roots: list[str | os.PathLike], expected: dict, limits: dict | None = None\n) -> dict:\n    """Select an exact authenticated package, deduplicating byte-identical copies."""\n\n    selected_limits = dict(DEFAULT_LIMITS)\n    selected_limits.update(limits or {})\n    candidates = _candidate_paths(search_roots, selected_limits)\n    accepted: list[dict] = []\n    rejected: list[dict] = []\n    for path in candidates:\n        try:\n            accepted.append(authenticate_candidate(path, expected, selected_limits))\n        except (AuthenticationError, OSError) as exc:\n            rejected.append({"path": str(path), "error": str(exc)})\n    if not accepted:\n        raise AuthenticationError(\n            "NO_MATCHING_PACKAGE; expected=" + json.dumps(expected, sort_keys=True) + "; rejected=" + json.dumps(rejected, sort_keys=True)\n        )\n    by_hash: dict[str, list[dict]] = {}\n    for row in accepted:\n        by_hash.setdefault(str(row["package_sha256"]), []).append(row)\n    if len(by_hash) != 1:\n        raise AuthenticationError(\n            "AMBIGUOUS_DIFFERENT_CONTENT; matches="\n            + json.dumps([{"path": row["path"], "sha256": row["package_sha256"]} for row in accepted], sort_keys=True)\n        )\n    selected = sorted(accepted, key=lambda row: str(row["path"]).casefold())[0]\n    selected["selection_status"] = (\n        "DUPLICATE_IDENTICAL_COPY_DEDUPED" if len(accepted) > 1 else "SELECTED_UNIQUE_VALID_PACKAGE"\n    )\n    selected["duplicate_paths"] = [row["path"] for row in sorted(accepted, key=lambda row: str(row["path"]).casefold())]\n    selected["candidate_count"] = len(candidates)\n    selected["rejected_candidates"] = rejected\n    return selected\n\n\ndef materialize_authenticated_package(authentication: dict, destination_value: str | os.PathLike) -> Path:\n    """Atomically materialize a previously authenticated package and reauthenticate it."""\n\n    source = Path(str(authentication["path"]))\n    destination = Path(destination_value)\n    expected = {\n        "expected_package_sha256": authentication["package_sha256"],\n        "expected_scientific_identity_hash": authentication["identity"].get("scientific_identity_hash"),\n        "expected_configuration_hash": authentication["identity"].get("configuration_hash"),\n        "expected_run_id": authentication["identity"].get("run_id"),\n        "expected_study_hash": authentication["identity"].get("study_hash"),\n        "expected_profile_id": authentication["identity"].get("profile_id"),\n        "expected_scale": authentication["identity"].get("scale"),\n        "expected_source_code_hash": authentication["source_code_hash"],\n        "expected_integrity_manifest": authentication["integrity_manifest"],\n        "expected_output_schema_version": authentication["identity"].get("output_schema_version"),\n        "expected_package_type": authentication["identity"].get("package_type"),\n        "expected_stage": authentication["identity"].get("stage"),\n    }\n    if authentication["form"] == "EXTRACTED_DIRECTORY":\n        authenticate_candidate(source, expected)\n        return source\n    if destination.exists():\n        marker = destination / ".source_sha256"\n        if destination.is_dir() and marker.is_file() and marker.read_text(encoding="utf-8").strip() == authentication["package_sha256"]:\n            directory_expected = {**expected, "expected_package_sha256": None}\n            authenticated_directory = authenticate_candidate(destination, directory_expected)\n            if authenticated_directory["identity"].get("scientific_identity_hash") == expected["expected_scientific_identity_hash"]:\n                return destination\n        raise FileExistsError("materialization destination contains a different or unauthenticated package")\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    partial = Path(tempfile.mkdtemp(prefix=f".{destination.name}.partial-", dir=destination.parent))\n    try:\n        with zipfile.ZipFile(source) as archive:\n            for info in archive.infolist():\n                target = partial.joinpath(*PurePosixPath(info.filename).parts)\n                if info.is_dir():\n                    target.mkdir(parents=True, exist_ok=True)\n                    continue\n                target.parent.mkdir(parents=True, exist_ok=True)\n                with archive.open(info) as source_handle, target.open("xb") as target_handle:\n                    shutil.copyfileobj(source_handle, target_handle, 1024 * 1024)\n        (partial / ".source_sha256").write_text(str(authentication["package_sha256"]) + "\\n", encoding="utf-8")\n        directory_expected = {**expected, "expected_package_sha256": None}\n        authenticate_candidate(partial, directory_expected)\n        os.replace(partial, destination)\n    except Exception:\n        shutil.rmtree(partial, ignore_errors=True)\n        raise\n    runtime = {\n        "schema_version": "certgen.runtime_location.v2",\n        "source_form": authentication["form"],\n        "source_sha256": authentication["package_sha256"],\n        "scientific_identity_hash": authentication["identity"].get("scientific_identity_hash"),\n        "source_code_hash": authentication["source_code_hash"],\n        "claim_allowed": False,\n    }\n    (destination / ".certgen_runtime_location.json").write_text(\n        json.dumps(runtime, indent=2, sort_keys=True) + "\\n", encoding="utf-8"\n    )\n    return destination\n\n\ndef authenticate_discover_materialize(\n    search_roots: list[str | os.PathLike], expected: dict, destination: str | os.PathLike\n) -> tuple[Path, dict]:\n    authentication = discover_authenticated_package(search_roots, expected)\n    root = materialize_authenticated_package(authentication, destination)\n    return root, authentication\n'
_TRUSTED_BOOTSTRAP_SHA256 = "1b790db1512a13136898fde403f0fba1acee07d03f906de0c7a7a30d10dcb8ab"
if hashlib.sha256(_TRUSTED_BOOTSTRAP_SOURCE.encode("utf-8")).hexdigest() != _TRUSTED_BOOTSTRAP_SHA256:
    raise RuntimeError("UNAUTHENTICATED_CODE_IMPORT_BLOCKED: trusted bootstrap source hash mismatch")
_BOOTSTRAP_NAMESPACE = {"__name__": "certgen_trusted_preimport_bootstrap"}
exec(compile(_TRUSTED_BOOTSTRAP_SOURCE, "<certgen-trusted-preimport-bootstrap>", "exec"), _BOOTSTRAP_NAMESPACE)

EXPECTED_PACKAGE_IDENTITY = json.loads('{"claim_allowed": false, "expected_configuration_hash": "daac551d32f196d5e944b350d4662a4bbf8472a617f8fbb805a813a5f3e812a0", "expected_integrity_manifest": "package_integrity_manifest.json", "expected_output_schema_version": "certgen.cvpr.preflight_output.v2", "expected_package_sha256": "d3a5b585383e12cfad82d94694fa1d8e2701de399617e8e515bafae57f33e93f", "expected_package_type": "PREFLIGHT_INPUT", "expected_profile_id": "cifar_integrity_minimal", "expected_run_id": "cifar10__checkpoint-preflight__tiny__29267d52e3c9", "expected_scale": null, "expected_scientific_identity_hash": "a31e6ba5c9486fdafd6491c6d7c03a2887049b0b47a8b2f2a1da0ba1a452e4a5", "expected_source_code_hash": "e2778da97acaa59913d544269c41be447017692786507382e073ed0f5ab982ca", "expected_stage": "preflight", "expected_study_hash": null, "schema_version": "certgen.expected_package_identity.v1"}')
if EXPECTED_PACKAGE_IDENTITY is None:
    raw_expected_identity = os.environ.get("CERTGEN_EXPECTED_PACKAGE_IDENTITY_JSON")
    if not raw_expected_identity:
        raise RuntimeError("explicit CERTGEN_EXPECTED_PACKAGE_IDENTITY_JSON is required; same-stage discovery is forbidden")
    EXPECTED_PACKAGE_IDENTITY = json.loads(raw_expected_identity)
if EXPECTED_PACKAGE_IDENTITY.get("schema_version") != "certgen.expected_package_identity.v1" or EXPECTED_PACKAGE_IDENTITY.get("claim_allowed") is not False:
    raise RuntimeError("invalid expected-package identity contract")
for _required_expected_field in (
    "expected_package_sha256", "expected_scientific_identity_hash", "expected_configuration_hash",
    "expected_run_id", "expected_source_code_hash", "expected_integrity_manifest",
    "expected_output_schema_version", "expected_package_type", "expected_stage",
):
    if not EXPECTED_PACKAGE_IDENTITY.get(_required_expected_field):
        raise RuntimeError(f"expected-package identity is missing {_required_expected_field}")
if EXPECTED_PACKAGE_IDENTITY["expected_package_type"] != "PREFLIGHT_INPUT" or EXPECTED_PACKAGE_IDENTITY["expected_stage"] != "preflight":
    raise RuntimeError("expected-package identity is for the wrong notebook stage")

INPUT_ROOT, AUTHENTICATION_REPORT = _BOOTSTRAP_NAMESPACE["authenticate_discover_materialize"](
    SEARCH_ROOTS,
    EXPECTED_PACKAGE_IDENTITY,
    "/kaggle/working/certgen-authenticated-input-preflight",
)
# This is intentionally the first point at which authenticated package code is importable.
sys.path.insert(0, str(INPUT_ROOT))
from certgen.notebooks.kaggle_io import load_frozen_configuration, verify_input_integrity
verify_input_integrity(INPUT_ROOT)
CONFIG = load_frozen_configuration(INPUT_ROOT)
WORK_ROOT = Path("/kaggle/working/certgen-cvpr")


## 3 Environment diagnostics


## 4 Dependency setup and validation

The bootstrap runs `python -m pip check` and writes `dependency_report.json`, `dependency_freeze.txt`, and `pip_check.txt`.


In [ ]:
from certgen.notebooks.environment_bootstrap import bootstrap_environment
ENVIRONMENT = bootstrap_environment(
    "kaggle_t4x2_preflight",
    output_dir=WORK_ROOT / "environment", network_allowed=bool(CONFIG["dependency_network_allowed"]), apply=True, install_mode=CONFIG.get("dependency_mode", "KAGGLE_INTERNET_ON_INSTALL"),
    search_roots=SEARCH_ROOTS,
    lock_path=INPUT_ROOT / "requirements/stage.lock",
    constraints_path=INPUT_ROOT / "requirements/kaggle-constraints.txt",
    lock_integrity_path=INPUT_ROOT / "requirements/lock_integrity.json",
    expected_input_identity={
        "package_sha256": AUTHENTICATION_REPORT["package_sha256"],
        "scientific_identity_hash": AUTHENTICATION_REPORT["identity"]["scientific_identity_hash"],
    },
)
if ENVIRONMENT["status"] != "ENVIRONMENT_COMPATIBLE":
    raise RuntimeError(ENVIRONMENT["restart_instruction"] or "environment incompatible")

if ENVIRONMENT["pip_check"]["returncode"] != 0:
    raise RuntimeError("python -m pip check failed; inspect pip_check.txt")


## 5 Asset discovery and validation


## 6 Configuration/provenance validation


In [ ]:
MODE = CONFIG["mode"]
if MODE not in {"resume", "restart", "force_new_run"}:
    raise ValueError("mode must be resume, restart, or force_new_run")


In [ ]:
from certgen.notebooks.model_assets import AssetPolicy
from certgen.notebooks.network_policy import network_policy_from_config
ASSET_POLICY = AssetPolicy(CONFIG["asset_policy"])
NETWORK_POLICY = network_policy_from_config(CONFIG)
if ASSET_POLICY is AssetPolicy.ONLINE_PREFLIGHT_DOWNLOAD and not NETWORK_POLICY.model_asset_network_allowed:
    raise RuntimeError("online preflight asset policy requires model asset network")
if CONFIG["kind"] != "preflight" and NETWORK_POLICY.model_asset_network_allowed:
    raise RuntimeError("model asset downloads are confined to checkpoint/extractor preflight")

from certgen.discovery import discover_asset_mount, write_asset_resolution_report
REQUIRED_ASSETS = {row["asset_id"]: row.get("revision") for row in CONFIG.get("assets", [])}
if REQUIRED_ASSETS:
    ASSET_RESOLUTION = discover_asset_mount(SEARCH_ROOTS, required_assets=REQUIRED_ASSETS)
    if ASSET_RESOLUTION["status"] not in {"SELECTED_UNIQUE_VALID_ASSET_MOUNT", "DUPLICATE_IDENTICAL_COPY_DEDUPED"}:
        raise RuntimeError(f"private asset discovery failed: {ASSET_RESOLUTION['status']}")
    ASSET_RESOLUTION_REPORT_PATH = WORK_ROOT / "asset_resolution_report.json"
    ASSET_RESOLUTION_REPORT = write_asset_resolution_report(ASSET_RESOLUTION, ASSET_RESOLUTION_REPORT_PATH)
    ASSET_RUNTIME_MAP = {row["asset_id"]: row for row in ASSET_RESOLUTION_REPORT["assets"]}
    ASSET_RUNTIME_BY_ID = {row["model_or_extractor_id"]: row for row in ASSET_RESOLUTION_REPORT["assets"]}
    ASSET_VALIDATION = ASSET_RESOLUTION
else:
    ASSET_RESOLUTION_REPORT_PATH = WORK_ROOT / "asset_resolution_report.json"
    ASSET_RUNTIME_MAP = {}
    ASSET_RUNTIME_BY_ID = {}


In [ ]:
from certgen.notebooks.kaggle_io import disk_guard
DISK = disk_guard("/kaggle/working", int(CONFIG.get("required_disk_bytes", 8 * 1024**3)))


In [ ]:
# Parent visibility check deliberately uses nvidia-smi and never imports or initializes PyTorch.
probe = subprocess.run(["nvidia-smi", "-L"], check=True, capture_output=True, text=True)
GPU_LINES = [line for line in probe.stdout.splitlines() if line.strip().startswith("GPU ")]
GPU_COUNT = len(GPU_LINES)
requested = int(CONFIG.get("requested_gpu_count", 2))
if GPU_COUNT < requested and not (GPU_COUNT == 1 and CONFIG.get("allow_single_gpu_fallback") is True):
    raise RuntimeError(f"requested {requested} GPUs but nvidia-smi reported {GPU_COUNT}")


## 7 Tiny dual-GPU dry run


In [ ]:
from certgen.notebooks.run_state import RunIdentity, prepare_run_directory
IDENTITY = RunIdentity(CONFIG["run_id"], CONFIG["configuration_hash"],
                       str(CONFIG.get("source_manifest_hash", CONFIG.get("reference_manifest_hash", "preflight_none"))),
                       str(CONFIG.get("asset_manifest_hash", "preflight_to_be_generated")))
RUN_STATE = prepare_run_directory(WORK_ROOT, IDENTITY, MODE)
RUN_ROOT = Path(RUN_STATE["run_dir"])
frozen_config_copy = RUN_ROOT / "configuration.yaml"
if frozen_config_copy.exists():
    if hashlib.sha256(frozen_config_copy.read_bytes()).hexdigest() != hashlib.sha256((INPUT_ROOT / "configuration.yaml").read_bytes()).hexdigest():
        raise ValueError("run-root frozen configuration differs from the uploaded configuration")
else:
    shutil.copy2(INPUT_ROOT / "configuration.yaml", frozen_config_copy)


In [ ]:
import multiprocessing as mp
mp.set_start_method("spawn", force=True)
from certgen.notebooks.subprocess_orchestrator import WorkerSpec, run_workers
TINY_SPECS = [
    WorkerSpec(
        worker_id=f"tiny_gpu_{gpu}",
        module="certgen.notebooks.workers.diagnostic_worker",
        physical_gpu=gpu,
        shard_id=f"tiny_gpu_{gpu}",
        args=("--out", str(RUN_ROOT / "tiny_gpu_diagnostic" / f"gpu_{gpu}"),
              "--configuration-hash", CONFIG["configuration_hash"],
              "--input-manifest-hash", str(CONFIG.get("input_manifest_hash", CONFIG.get("reference_manifest_hash", "diagnostic_static_input")))),
        completion_marker=str(RUN_ROOT / "tiny_gpu_diagnostic" / f"gpu_{gpu}" / "worker_completion.json"),
        configuration_hash=CONFIG["configuration_hash"],
        input_manifest_hash=str(CONFIG.get("input_manifest_hash", CONFIG.get("reference_manifest_hash", "diagnostic_static_input"))),
        asset_manifest_hash="no_assets_required",
        worker_type="diagnostic",
        config_schema_version="certgen.kaggle.diagnostic_config.v1",
        output_schema_version="certgen.kaggle.diagnostic_output.v1",
    )
    for gpu in range(2)
]
TINY_DUAL_GPU = run_workers(TINY_SPECS, output_dir=RUN_ROOT / "tiny_gpu_orchestration", resume=MODE == "resume")
if sorted(row["physical_gpu"] for row in TINY_DUAL_GPU["workers"]) != [0, 1]:
    raise RuntimeError("tiny dry run did not execute one worker on each physical GPU")


## 8 Runtime calibration


In [ ]:
from certgen.cvpr.contracts import atomic_write_json
CALIBRATION_ROWS = [
    json.loads((RUN_ROOT / "tiny_gpu_diagnostic" / f"gpu_{gpu}" / "diagnostic_report.json").read_text(encoding="utf-8"))
    for gpu in range(2)
]
RUNTIME_CALIBRATION = {
    "model_load_seconds": [row["model_load_seconds"] for row in CALIBRATION_ROWS],
    "warmup_seconds": [row["warmup_seconds"] for row in CALIBRATION_ROWS],
    "throughput_iterations_per_second": [row["throughput_iterations_per_second"] for row in CALIBRATION_ROWS],
    "peak_vram_bytes": [row["peak_allocated_bytes"] for row in CALIBRATION_ROWS],
    "safe_batch_size": min(row["safe_batch_size"] for row in CALIBRATION_ROWS),
    "planning_only": True,
    "not_empirical_evidence": True,
    "claim_allowed": False,
}
atomic_write_json(RUNTIME_CALIBRATION, RUN_ROOT / "runtime_calibration.json")


## 9 Full parallel execution


In [ ]:
from certgen.notebooks.subprocess_orchestrator import WorkerSpec

specs = []
for index, asset in enumerate(CONFIG["assets"]):
    gpu = index % GPU_COUNT
    resolved_asset = ASSET_RUNTIME_MAP[asset["asset_id"]]
    cache_root = Path(resolved_asset["snapshot_root"])
    if asset["asset_kind"] == "model":
        model_id = asset["model_or_extractor_id"]
        shard_id = f"model__{model_id}"
        worker_out = RUN_ROOT / "per_model" / model_id
        specs.append(WorkerSpec(
            worker_id=shard_id, module="certgen.notebooks.workers.preflight_worker",
            physical_gpu=gpu, shard_id=shard_id,
            args=("--config", str(INPUT_ROOT / "configuration.yaml"), "--asset-id", asset["asset_id"],
                  "--shard-id", shard_id, "--cache-root", str(cache_root),
                  "--asset-resolution-report", str(ASSET_RESOLUTION_REPORT_PATH), "--out", str(worker_out)),
            completion_marker=str(worker_out / "worker_completion.json"),
            configuration_hash=CONFIG["configuration_hash"], input_manifest_hash=CONFIG["input_manifest_hash"],
        ))
    else:
        extractor_id = asset["model_or_extractor_id"]
        asset_shard = f"extractor_asset__{extractor_id}"
        asset_out = RUN_ROOT / "per_asset" / extractor_id
        specs.append(WorkerSpec(
            worker_id=asset_shard, module="certgen.notebooks.workers.preflight_worker",
            physical_gpu=gpu, shard_id=asset_shard,
            args=("--config", str(INPUT_ROOT / "configuration.yaml"), "--asset-id", asset["asset_id"],
                  "--shard-id", asset_shard, "--cache-root", str(cache_root),
                  "--asset-resolution-report", str(ASSET_RESOLUTION_REPORT_PATH), "--out", str(asset_out), "--asset-only"),
            completion_marker=str(asset_out / "worker_completion.json"),
            configuration_hash=CONFIG["configuration_hash"], input_manifest_hash=CONFIG["input_manifest_hash"],
        ))
        extractor_shard = f"extractor__{extractor_id}"
        extractor_out = RUN_ROOT / "per_extractor" / extractor_id
        specs.append(WorkerSpec(
            worker_id=extractor_shard, module="certgen.notebooks.workers.extractor_preflight_worker",
            physical_gpu=gpu, shard_id=extractor_shard,
            args=("--config", str(INPUT_ROOT / "configuration.yaml"), "--extractor-id", extractor_id,
                  "--shard-id", extractor_shard, "--asset-manifest", str(asset_out / "asset_manifest.json"),
                  "--cache-root", str(cache_root), "--out", str(extractor_out)),
            completion_marker=str(extractor_out / "worker_completion.json"),
            configuration_hash=CONFIG["configuration_hash"], input_manifest_hash=CONFIG["input_manifest_hash"],
        ))

from certgen.notebooks.kaggle_io import assert_unique_shards
assert_unique_shards([{"shard_id": spec.shard_id} for spec in specs])


In [ ]:
from certgen.notebooks.subprocess_orchestrator import run_workers
ORCHESTRATION = run_workers(specs, output_dir=RUN_ROOT / "orchestration", timeout_seconds=CONFIG.get("worker_timeout_seconds"), resume=MODE == "resume")


In [ ]:
FAILED = [row for row in ORCHESTRATION["workers"] if row["status"] not in {"COMPLETE", "REUSED_VALID_COMPLETION"}]
for row in ORCHESTRATION["workers"]:
    print(row["worker_id"], row["status"], row.get("log"), row.get("rerun_command"))
if FAILED:
    raise RuntimeError("BLOCKED_PARTIAL_FAILURE; preserve completed shards and use the emitted rerun commands")


## 10 Merge and validation


In [ ]:
from certgen.cvpr.contracts import atomic_write_json
from certgen.notebooks.kaggle_io import all_worker_statuses_complete
if not all_worker_statuses_complete(ORCHESTRATION):
    raise RuntimeError("shard validation failed")
ROOT_STATUS = {"status_code": "PREFLIGHT_PASS", "passed": True, "configuration_hash": CONFIG["configuration_hash"],
                "mode": MODE, "expected_workers": sorted(spec.worker_id for spec in specs),
                "completed_workers": sorted(row["worker_id"] for row in ORCHESTRATION["workers"]),
                "output_schema_version": CONFIG["output_schema_version"],
                "evidence_class": "non_evidence_preflight", "claim_allowed": False}
atomic_write_json(ROOT_STATUS, RUN_ROOT / "status.json")
if CONFIG["kind"] == "preflight":
    ROOT_STATUS["results"] = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(RUN_ROOT.glob("per_model/*/status.json"))]
    ROOT_STATUS["extractor_results"] = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(RUN_ROOT.glob("per_extractor/*/status.json"))]
    atomic_write_json(ROOT_STATUS, RUN_ROOT / "checkpoint_preflight_status.json")
elif CONFIG["kind"] == "generation":
    atomic_write_json(ROOT_STATUS, RUN_ROOT / "generation_status.json")
else:
    atomic_write_json(ROOT_STATUS, RUN_ROOT / "feature_extraction_status.json")


In [ ]:
# Deterministic merge is sample-ID based inside each worker and orchestration status is sorted by worker ID.
MERGE_INDEX = sorted((row["worker_id"], row["shard_id"]) for row in ORCHESTRATION["workers"])
atomic_write_json({"workers": MERGE_INDEX, "configuration_hash": CONFIG["configuration_hash"], "claim_allowed": False}, RUN_ROOT / "merge_index.json")


In [ ]:
from certgen.notebooks.kaggle_io import write_integrity_manifest
INTEGRITY = write_integrity_manifest(RUN_ROOT)


## 11 Atomic output ZIP


In [ ]:
from certgen.notebooks.kaggle_io import copyback_instructions
from certgen.notebooks.final_zip import finalize_output_zip, validate_final_zip, write_multipart_fallback
ZIP_PATH = Path("/kaggle/working") / f"certgen_cvpr_preflight_{CONFIG['run_id']}.zip"
(RUN_ROOT / "copyback_instructions.md").write_text(copyback_instructions("preflight", ZIP_PATH), encoding="utf-8")
write_integrity_manifest(RUN_ROOT)
ZIP = finalize_output_zip(RUN_ROOT, ZIP_PATH, mode=MODE, configuration_hash=CONFIG["configuration_hash"],
                          asset_manifest_hash=str(CONFIG.get("asset_manifest_hash", "preflight_generated")),
                          input_identity={"package_sha256": AUTHENTICATION_REPORT["package_sha256"],
                                          "scientific_identity_hash": AUTHENTICATION_REPORT["identity"]["scientific_identity_hash"]})

if not validate_final_zip(RUN_ROOT, ZIP_PATH)["passed"]:
    raise RuntimeError("final output ZIP revalidation failed")
MULTIPART = write_multipart_fallback(ZIP_PATH) if ZIP_PATH.stat().st_size > 3800 * 1024**2 else None


## 12 Local handoff


## Copy-back and local import

Copy the final ZIP without unpacking it; renaming is allowed. Preserve its hash and run either:

`python3 -m certgen import preflight <copied-back-zip>`

or `python3 scripts/run_all_available_cpu_stages.py --resume --explain --search-root /path/to/downloads` for recursive content-based resume.



On failure, preserve completed shards/logs and run only each exact `rerun_command` emitted in the monitoring cell. Changed input, config, or asset hashes require `restart` or `force_new_run`; never reuse incompatible markers.


In [ ]:
FINAL_STATUS = {"status": "RUN_READY_BY_LOCAL_CONTRACT_REAL_KAGGLE_EXECUTION_REQUIRED",
                "output_zip": ZIP, "evidence_class": "non_evidence_preflight", "claim_allowed": False}
print(json.dumps(FINAL_STATUS, indent=2, sort_keys=True))
